In [5]:
import pandas as pd
import json
import os
import random
import sys


sys.path.append('/Users/danylo/Desktop/coursework/lab4/src')
from rules import extract_all

DATA_PATH = '/Users/danylo/Desktop/coursework/lab4/data/processed_v2.csv' 
OUTPUT_PATH = '/Users/danylo/Desktop/coursework/lab4/data/sample/lab4_gold_ie.jsonl'

def generate_sample():
    print("Завантаження даних...")
    df = pd.read_csv(DATA_PATH)
    
    texts = df['premise_v2'].dropna().unique().tolist()
    
    random.seed(111)
    sample_texts = random.sample(texts, min(70, len(texts)))
    
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    
    extracted_count = 0
    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        for i, text in enumerate(sample_texts):
            entities = extract_all(text)
            
            for ent in entities:
                record = {
                    "text_id": f"gold_{i}",
                    "text": text,
                    "field_type": ent["field_type"],
                    "span_text": text[ent["start_char"]:ent["end_char"]], 
                    "start_char": ent["start_char"],
                    "end_char": ent["end_char"],
                    "normalized_value": ent["value"],
                    "method": ent["method"],
                    "is_correct": True 
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')
                extracted_count += 1
                
    print(f"Знайдено {extracted_count} сутностей у 50 реченнях.")
    print(f"Файл збережено: {OUTPUT_PATH}")
    
if __name__ == "__main__":
    generate_sample()

Завантаження даних...
Знайдено 71 сутностей у 50 реченнях.
Файл збережено: /Users/danylo/Desktop/coursework/lab4/data/sample/lab4_gold_ie.jsonl


In [6]:
import pandas as pd
import json

GOLD_FILE_PATH = '/Users/danylo/Desktop/coursework/lab4/data/sample/lab4_gold_ie.jsonl' 

data = []
try:
    with open(GOLD_FILE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
            
    df_gold = pd.DataFrame(data)

    metrics = []
    for f_type in df_gold['field_type'].unique():
        subset = df_gold[df_gold['field_type'] == f_type]
        all_ext = len(subset)
        
        correct_ext = subset['is_correct'].sum() 
        
        precision = correct_ext / all_ext if all_ext > 0 else 0
        
        metrics.append({
            "Field Type": f_type,
            "Correct (TP)": correct_ext,
            "All Extractions": all_ext,
            "Precision": f"{precision:.2%}"
        })

    df_metrics = pd.DataFrame(metrics)
    print("=== ТАБЛИЦЯ PRECISION ===")
    display(df_metrics)

except FileNotFoundError:
    print(f"Файл {GOLD_FILE_PATH} не знайдено.")
except KeyError:
    print("Помилка: no is_correct")

=== ТАБЛИЦЯ PRECISION ===


,Field Type,Correct (TP),All Extractions,Precision
0,COLOR,54,57,94.74%
1,LOCATION,20,22,90.91%
2,QUANTITY,30,36,83.33%


**Аналіз помилок:**
1. **Омонімія коренів (COLOR / LOCATION):** Корінь кольору `біл-` спрацьовував на слова типу "біличах" (артефакт перекладу) або "більярд", а корінь `пол-` (від "пол" / підлога) спрацював на слово "полосовій" (візерунок одягу). Також правило LOCATION спіймало прикметник "лісовій" замість самої локації.
2. **Займенники (QUANTITY):** Більшість помилок у витягу кількостей пов'язані зі словом "один". Регулярний вираз витягує його як цифру `1`, проте в контексті ("один з команди", "один з яких курить", "бачить ззаду один") це слово виконує роль займенника.
3. **Кількісні прислівники (QUANTITY):** Слово "багато" було витягнуто як кількість, але у фразі "багато снігує" воно є прислівником міри дії, а не лічильником об'єктів.

In [8]:
import json

EDGE_CASES_PATH = '../tests/ie_edge_cases.jsonl'

print("=== ЗАПУСК ТЕСТІВ НА EDGE CASES ===\n")

try:
    with open(EDGE_CASES_PATH, 'r', encoding='utf-8') as f:
        edge_cases = [json.loads(line) for line in f]

    passed = 0
    for case in edge_cases:
        text = case['raw_text']
        target_field = case['field_type']
        
        extracted = extract_all(text)
        
        found_of_type = [e for e in extracted if e['field_type'] == target_field]
        
        print(f"ID: {case['id']}")
        print(f"Текст: {text}")
        print(f"Очікувана поведінка: {case['expected_behavior']}")
        
        if found_of_type:
            vals = [str(e['value']) for e in found_of_type]
            print(f"Знайдено: {', '.join(vals)}")
        else:
            print(f"Не знайдено")
        
        print("-" * 50)
        passed += 1

    print(f"\nУспішно протестовано {passed} кейсів.")

except FileNotFoundError:
    print(f"Файл {EDGE_CASES_PATH} не знайдено.")

=== ЗАПУСК ТЕСТІВ НА EDGE CASES ===

ID: edge_q_01
Текст: Один з команди пішов додому.
Очікувана поведінка: Не вважати 'Один' кількістю, оскільки в цьому контексті це займенник.
Знайдено: 1
--------------------------------------------------
ID: edge_q_02
Текст: Хлопчик 5 років грає з м'ячем.
Очікувана поведінка: Ігнорувати '5', оскільки це характеристика віку, а не лічильник об'єктів.
Знайдено: 5
--------------------------------------------------
ID: edge_q_03
Текст: Він виконав роботу на 100%.
Очікувана поведінка: Ігнорувати '100', оскільки це відсотки, а не кількість людей чи предметів.
Знайдено: 100
--------------------------------------------------
ID: edge_q_04
Текст: Вони живуть у будівлі 5.
Очікувана поведінка: Ігнорувати '5', оскільки це номер будівлі (ідентифікатор), а не їх кількість.
Знайдено: 5
--------------------------------------------------
ID: edge_q_05
Текст: Зустріч розпочнеться о 3 годині.
Очікувана поведінка: Не вважати '3' кількістю, оскільки це маркер часу.
Знайд